In [ ]:
using Pkg
Pkg.activate("/Users/bursche/Documents/GitHub/JPEC_BCRIT")
Base.active_project()

using GeneralizedPerturbedEquilibrium
using GeneralizedPerturbedEquilibrium: Analysis
using Printf

using Plots
default(
    fontfamily="Georgia",
    margin=12Plots.mm,
    size=(800, 500),
    dpi=150
)

In [ ]:
h5path = "/Users/bursche/Documents/GitHub/JPEC_BCRIT/examples/DIIID-like_SLAYER_example/gpec.h5"

In [ ]:
using HDF5

h5open(h5path, "r") do file
    println("Top-level keys:")
    println(collect(keys(file)))

    if haskey(file, "Tearing")
        println("\nKeys in Tearing:")
        println(collect(keys(file["Tearing"])))

        if haskey(file["Tearing"], "CriticalResonantField")
            println("\nKeys in Tearing/CriticalResonantField:")
            println(collect(keys(file["Tearing"]["CriticalResonantField"])))
        end
    else
        println("No Tearing group found.")
    end
end

In [ ]:
# Print the contents of the Tearing/CriticalResonantField group
h5open(h5path, "r") do file
    if haskey(file, "Tearing") && haskey(file["Tearing"], "CriticalResonantField")
        println("\nContents of Tearing/CriticalResonantField:")
        # Iterate over the keys in the Tearing/CriticalResonantField group and print their names and types
        for key in keys(file["Tearing"]["CriticalResonantField"])
            dataset = file["Tearing"]["CriticalResonantField"][key]
            println("Key: $key, Type: $(typeof(dataset))")
            # print data 
            if isa(dataset, HDF5.Dataset)
                data = read(dataset)
                println("Data: $data")
            end
        end
    else
        println("No Tearing/CriticalResonantField group found.")
    end
    # Print bcrit data as .2e (in one line)
    println("\nBcrit data: $(join([@sprintf("%.2e", x) for x in read(file["Tearing"]["CriticalResonantField"]["br_crit"])], " "))")
end



In [ ]:
println(keys(h5open(h5path, "r")["Tearing"]["CriticalResonantField"]["Scan"]["surface_1"]))

In [ ]:
function bcrit_diag_plots(Δs, Qs, bal,name)

    p1 = plot(Qs, imag.(Δs), label="Im(Δ)", lw=2)
    #plot!(p1, Qs, real.(Δs), label="Re(Δ)", lw=2)
    xlabel!(p1, "Q")
    ylabel!(p1, "Δ")
    title!(p1, "Inner-layer Δ(Q) - $name")

    p2 = plot(Qs, real.(bal), label="Re(balance)", lw=2)
    plot!(p2, Qs, imag.(bal), label="Im(balance)", lw=2)
    xlabel!(p2, "Q")
    ylabel!(p2, "balance")
    title!(p2, "2P(Q0-Q)/jxb - $name")

    plot(p1, p2, layout=(2,1), size=(800, 1000))


end

In [ ]:
# print all P values
println("\nP values for each surface in the scan:")
h5open(h5path, "r") do file
    for i in 1:6
        ss = "surface_$i"
        if haskey(file["Tearing"]["CriticalResonantField"]["Scan"], ss)
            P = read(file["Tearing"]["CriticalResonantField"]["Scan"][ss]["P"])
            println("Surface $i: P = $(join([@sprintf("%.2e", x) for x in P], " "))")
        else
            println("Surface $i: No data found.")
        end
    end
end
# want to print chi_tor for each surface in the scan, need to use P = (4π * 1e-7) .* abs.(chi_sel) ./ η_neo, so chi_sel = P * η_neo / (4π * 1e-7)
println("\nchi_tor values for each surface in the scan:")
h5open(h5path, "r") do file
    for i in 1:6
        ss = "surface_$i"
        if haskey(file["Tearing"]["CriticalResonantField"]["Scan"], ss)
            P = read(file["Tearing"]["CriticalResonantField"]["Scan"][ss]["P"])
            η_neo = read(file["Tearing"]["PerSurface"]["eta"])
            chi_tor = P .* η_neo[i] ./ (4π * 1e-7)
            println("Surface $i: chi_tor = $(join([@sprintf("%.2e", x) for x in chi_tor], " "))")   
        else
            println("Surface $i: No data found.")
        end
    end
end

In [ ]:
# plot jxb vs Qs for surface 1
# jxbs = [-imag(1.0 / (d + 1e-2)) for d in Δs]
surface_num = 2
h5open(h5path, "r") do file
    Δs = read(file["Tearing"]["CriticalResonantField"]["Scan"]["surface_$(surface_num)"]["delta"])
    Qs = read(file["Tearing"]["CriticalResonantField"]["Scan"]["surface_$(surface_num)"]["Q"])
    bal = read(file["Tearing"]["CriticalResonantField"]["Scan"]["surface_$(surface_num)"]["balance"])
    jxbs = [-imag(1.0 / (d + 1e-2)) for d in Δs]
    aplot = plot(Qs, jxbs, label="jxb", lw=2)
    xlabel!("Q")
    ylabel!("jxb")
    title!("jxb vs Q for surface $(surface_num)")
    println("max of balance: ", maximum(bal))
    println("max of jxb: ", maximum(jxbs))
    println("bcrit: ", read(file["Tearing"]["CriticalResonantField"]["br_crit"]))
    println("Qpeak: ", read(file["Tearing"]["CriticalResonantField"]["Qpeak"]))
    println("Q_e:   ", read(file["Tearing"]["PerSurface"]["Q_e"]))
    # plot balance vs q with bcrit as vertical line
    p2 = plot(Qs, real.(bal), label="Re(balance)", lw=2)
    plot!(p2, Qs, imag.(bal), label="Im(balance)", lw=2)
    xlabel!(p2, "Q")
    ylabel!(p2, "balance")
    title!(p2, "2P(Q0-Q)/jxb - surface $(surface_num)")
    vline!(p2, [read(file["Tearing"]["CriticalResonantField"]["br_crit"])], label="bcrit", lw=2, lc=:red)
    # add Q_e as vertical line
    vline!(p2, [read(file["Tearing"]["PerSurface"]["Q_e"])[surface_num]], label="Q_e", lw=2, lc=:green)
    vline!(p2, [read(file["Tearing"]["PerSurface"]["Q_i"])[surface_num]], label="Q_i", lw=2, lc=:blue)

    display(aplot)
    display(p2)

end

"""
Local maxima indices: [104, 103] with values: [80.15320925995013, 79.66428209429431] and corresponding Qs: [0.35175879396984927, 0.25125628140703515] closest Q to Q_e: -0.35175879396984927 closest Q to Q_i: 0.35175879396984927
Warning: The first local maximum may correspond to an electron or ion diamagnetic resonance. Selecting the second local maximum instead.
Selected local maximum index: 103 with value: 79.66428209429431 and corresponding Q: 0.25125628140703515
"""

In [ ]:
# Plot bcrit vs q surface

# print all contents of bcrit /scan /surface 1/
h5open(h5path, "r") do file
    bcrit = file["Tearing"]["CriticalResonantField"]

    for i in 1:6
        ss = "surface_$i"

        delta  = read(bcrit["Scan"][ss]["delta"])
        Q      = read(bcrit["Scan"][ss]["Q"])
        balance = read(bcrit["Scan"][ss]["balance"])

        display(bcrit_diag_plots(delta, Q, balance, "surface_$i"))
    end
end

In [ ]:
p_eq = Analysis.Equilibrium.plot_equilibrium_summary(h5path)
p_ffs = Analysis.ForceFreeStates.plot_ffs_summary(h5path)
#p_pe = Analysis.PerturbedEquilibrium.plot_perturbed_equilibrium_summary(h5path)

display(p_eq)
display(p_ffs)
#display(p_pe)